In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# ==========================================
# 0. Lectura del Superdataset de precios horarios de la electricidad, generacion y climatología
# ==========================================
df_dummy = pd.read_csv('ruta/a/tu_dataset.csv') #TODO!
# ==========================================
# 1. PREPARACIÓN Y DIVISIÓN TEMPORAL
# ==========================================
X = df_dummy.drop(columns=['price', 'datetime'])
y = df_dummy['price']

# División cronológica estricta: entrenar con 2020-2023, validar con 2024 completo
X_train, X_test = X[X['year'] < 2024], X[X['year'] == 2024]
y_train, y_test = y[X['year'] < 2024], y[X['year'] == 2024]

# Eliminamos 'year' de los predictores porque los árboles no extrapolan bien hacia el futuro
X_train = X_train.drop(columns=['year'])
X_test = X_test.drop(columns=['year'])

# Variables categóricas temporales identificadas
cat_features = ['hour', 'month', 'dayofweek', 'is_weekend']


# ==========================================
# 2. CONFIGURACIÓN LIGHTGBM (Anti-Overfitting)
# ==========================================
# Explicación de parámetros clave:
# - max_depth y num_leaves: Controlan el tamaño del árbol. Valores bajos evitan que memorice datos aislados.
# - min_data_in_leaf: Exige que cada conclusión matemática agrupe al menos 100 horas del histórico.
# - learning_rate bajo + n_estimators alto con early stopping: El modelo aprende de forma muy gradual.
# - reg_lambda: Regularización L2 para suavizar el impacto del precio del gas de 2022.
# - colsample_bytree: Cada árbol solo ve el 70% de las variables climáticas, evitando obsesionarse con una sola.

model_lgb = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.015,         # Aprendizaje lento y seguro
    max_depth=6,                 # Árboles poco profundos
    num_leaves=35,               # Límite de nodos terminales
    min_data_in_leaf=100,        # Evita hojas con pocas horas registradas
    reg_lambda=10.0,             # Penalización L2 estricta contra picos artificiales
    reg_alpha=2.0,               # Penalización L1 para descartar ruido climático
    colsample_bytree=0.7,        # Submuestreo de columnas por árbol
    subsample=0.8,               # Cada árbol usa solo el 80% de los días del histórico
    random_state=42,
    n_jobs=-1
)

model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)] # Frena si el error en 2024 empieza a subir
)


# ==========================================
# 3. CONFIGURACIÓN XGBOOST (Anti-Overfitting)
# ==========================================
# Explicación de parámetros clave:
# - max_depth=5: Restringe drásticamente la estructura jerárquica del árbol (crecimiento por niveles).
# - min_child_weight: Equivalente al número de muestras en nodos. Si el peso es bajo, descarta la división.
# - gamma: Penalización por realizar cualquier división nueva en el árbol. Exige una ganancia de calidad alta.

model_xgb = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.015,
    max_depth=5,                 # Muy controlado para evitar memorizar curvas diarias exactas
    min_child_weight=50,         # Hojas con alta representación de datos
    gamma=5.0,                   # Penalización estricta por complejidad del árbol
    reg_lambda=15.0,             # Alta regularización L2 para absorber shocks de precios fósiles
    reg_alpha=2.0,               # Regularización L1
    colsample_bytree=0.7,        # Fuerza a buscar alternativas a las variables climáticas dominantes
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50     # Parámetro nativo de parada temprana en el constructor o fit
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)


# ==========================================
# 4. CONFIGURACIÓN CATBOOST (Anti-Overfitting)
# ==========================================
# Explicación de parámetros clave:
# - depth=6: Los árboles simétricos de CatBoost con profundidad 6 son sumamente estables.
# - l2_leaf_reg: Parámetro nativo de regularización L2.
# - rsm (Random Subspace Method): Equivalente a colsample_bytree, controla el porcentaje de variables por corte.

model_cat = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.02,
    depth=6,                     # Estructura simétrica robusta ante outliers climáticos
    l2_leaf_reg=12.0,            # Fuerte control de magnitudes en las hojas
    rsm=0.7,                     # Selecciona aleatoriamente subconjuntos de tus 27 columnas
    subsample=0.8,
    random_state=42,
    verbose=0,
    early_stopping_rounds=50
)

# Para CatBoost especificamos las columnas temporales que actúan como categorías de negocio
model_cat.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test)
)


# ==========================================
# 5. EVALUACIÓN COMPARATIVA EN EL AÑO TEST (2024)
# ==========================================
modelos = {'LightGBM': model_lgb, 'XGBoost': model_xgb, 'CatBoost': model_cat}

print("\n=== RENDIMIENTO DE LOS MODELOS REGULARIZADOS EN EL AÑO TEST (2024) ===")
for nombre, modelo in modelos.items():
    preds = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    print(f"[{nombre}] MAE: {mae:.3f} | RMSE: {rmse:.3f}")



# ==========================================
# 6. EXPORTAR Y GUARDAR RESULTADOS (LOS 3 MODELOS)
# ==========================================
# Extraemos las predicciones de cada modelo
preds_lgb = model_lgb.predict(X_test)
preds_xgb = model_xgb.predict(X_test)
preds_cat = model_cat.predict(X_test)

# Creamos un nuevo dataframe para comparar la realidad vs la predicción
df_resultados = pd.DataFrame({
    'Precio_Real': y_test,
    'Prediccion_LightGBM': preds_lgb,
    'Diferencia_LightGBM': y_test - preds_lgb,
    'Prediccion_XGBoost': preds_xgb,
    'Diferencia_XGBoost': y_test - preds_xgb,
    'Prediccion_CatBoost': preds_cat,
    'Diferencia_CatBoost': y_test - preds_cat
}, index=y_test.index) # Usamos el índice de fechas que preparamos en el Paso 1

# Guardamos los resultados en un archivo Excel o CSV para que puedas abrirlo
df_resultados.to_csv('predicciones_2024.csv')
print("\n¡Archivo 'predicciones_2024.csv' generado con éxito!")

In [ ]:
import matplotlib.pyplot as plt

# ==========================================
# 7. VISUALIZACIÓN DE RESULTADOS (LOS 3 MODELOS)
# ==========================================
# Filtramos un tramo corto (ej. primera quincena de enero de 2024)
df_plot = df_resultados.loc['2024-01-01':'2024-01-15']

plt.figure(figsize=(16, 7))

# 1. Línea del Precio Real (Azul, continua y más gruesa para que destaque)
plt.plot(df_plot.index, df_plot['Precio_Real'], 
         label='Precio Real', color='blue', linewidth=3)

# 2. Línea LightGBM (Verde, punteada)
plt.plot(df_plot.index, df_plot['Prediccion_LightGBM'], 
         label='LightGBM', color='green', linestyle='--', linewidth=1.5, alpha=0.8)

# 3. Línea XGBoost (Rojo, punteada)
plt.plot(df_plot.index, df_plot['Prediccion_XGBoost'], 
         label='XGBoost', color='red', linestyle='--', linewidth=1.5, alpha=0.8)

# 4. Línea CatBoost (Naranja, punteada)
plt.plot(df_plot.index, df_plot['Prediccion_CatBoost'], 
         label='CatBoost', color='darkorange', linestyle='--', linewidth=1.5, alpha=0.8)

# Añadimos títulos y diseño
plt.title('Comparativa de Modelos: Precio Real vs Predicciones (Enero 2024)', fontsize=14, fontweight='bold')
plt.xlabel('Fecha y Hora', fontsize=12)
plt.ylabel('Precio (€/MWh)', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# Guardamos y mostramos
plt.savefig('grafico_comparativa_3modelos_enero_2024.png', dpi=300)
plt.show()
